In [1]:
import os

data_path = "/kaggle/input/datasets/vishwasmishra1234/trash-net"

for folder in os.listdir(data_path):
    print(folder)
for folder in os.listdir(data_path):
    folder_path = os.path.join(data_path, folder)
    if os.path.isdir(folder_path):
        print(folder, ":", len(os.listdir(folder_path)))

train_imgEFFICIENTNET
val_imgEFFICIENTNET
train_imgEFFICIENTNET : 4
val_imgEFFICIENTNET : 4


In [2]:
import os
import matplotlib.pyplot as plt
import cv2
import random

def show_images(split, class_name):
    folder = os.path.join(data_path, split, class_name)
    
    print("Using path:", folder)  # debug (important)
    
    if not os.path.exists(folder):
        print("❌ Wrong path. Check split/class name.")
        return
    
    images = os.listdir(folder)
    
    plt.figure(figsize=(10,5))
    
    for i in range(5):
        img_path = os.path.join(folder, random.choice(images))
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        plt.subplot(1,5,i+1)
        plt.imshow(img)
        plt.axis("off")
    
    plt.suptitle(f"{split} - {class_name}")
    plt.show()

In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_path = "/kaggle/input/datasets/vishwasmishra1234/trash-net/train_imgEFFICIENTNET"
val_path = "/kaggle/input/datasets/vishwasmishra1234/trash-net/val_imgEFFICIENTNET"

In [4]:
#data augmentation for increasing dataset
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomRotation(10),
    
    transforms.ToTensor(),  # ✅ convert to tensor FIRST
    
    transforms.RandomErasing(p=0.3),
])

In [5]:
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [6]:
train_dataset = datasets.ImageFolder(train_path, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_path, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [7]:
print(train_dataset.classes)

['glass', 'metal', 'other', 'plastic']


In [8]:
import timm
import torch.nn as nn

model = timm.create_model('efficientnet_b0', pretrained=True)

# Modify final layer
model.classifier = nn.Linear(model.classifier.in_features, 4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [9]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [10]:
num_epochs = 10

best_val_acc = 0
patience = 2
counter = 0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    # ---- TRAIN ----
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_acc = 100 * correct / total
    
    # ---- VALIDATION ----
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_acc = 100 * correct / total
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    # ---- EARLY STOPPING ----
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        counter = 0
    else:
        counter += 1
    
    if counter >= patience:
        print("Early stopping triggered")
        break


Epoch 1/10
Train Loss: 156.2258 | Train Acc: 75.81%
Val Loss: 13.6317 | Val Acc: 92.98%

Epoch 2/10
Train Loss: 58.5590 | Train Acc: 91.22%
Val Loss: 7.0397 | Val Acc: 96.44%

Epoch 3/10
Train Loss: 37.5141 | Train Acc: 94.42%
Val Loss: 5.3350 | Val Acc: 97.08%

Epoch 4/10
Train Loss: 26.8809 | Train Acc: 96.15%
Val Loss: 3.7324 | Val Acc: 97.95%

Epoch 5/10
Train Loss: 20.0530 | Train Acc: 97.18%
Val Loss: 2.9095 | Val Acc: 98.38%

Epoch 6/10
Train Loss: 18.9853 | Train Acc: 97.45%
Val Loss: 2.8228 | Val Acc: 98.06%

Epoch 7/10
Train Loss: 15.6478 | Train Acc: 97.83%
Val Loss: 2.5616 | Val Acc: 98.49%

Epoch 8/10
Train Loss: 13.8658 | Train Acc: 98.13%
Val Loss: 2.3014 | Val Acc: 98.81%

Epoch 9/10
Train Loss: 13.0316 | Train Acc: 98.24%
Val Loss: 2.1702 | Val Acc: 98.81%

Epoch 10/10
Train Loss: 10.3762 | Train Acc: 98.43%
Val Loss: 2.4782 | Val Acc: 98.60%
Early stopping triggered
